# Complete Guide to Neural Networks
## Types, Architecture, Working Process, Advantages, Disadvantages & Use Cases

---

## Table of Contents

1. [Feedforward Neural Networks (FNN)](#fnn)
2. [Convolutional Neural Networks (CNN)](#cnn)
3. [Recurrent Neural Networks (RNN)](#rnn)
4. [Long Short-Term Memory (LSTM)](#lstm)
5. [Gated Recurrent Unit (GRU)](#gru)
6. [Autoencoders](#autoencoder)
7. [Variational Autoencoders (VAE)](#vae)
8. [Generative Adversarial Networks (GAN)](#gan)
9. [Transformer Networks](#transformer)
10. [Bidirectional RNN & Attention](#attention)
11. [Graph Neural Networks (GNN)](#gnn)
12. [Capsule Networks](#capsule)
13. [Boltzmann Machines](#boltzmann)
14. [Comparison & Selection Guide](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from sklearn.datasets import make_circles, make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

---
# 1. Feedforward Neural Networks (FNN) {#fnn}

## Architecture Overview
```
Input Layer → Hidden Layer 1 → Hidden Layer 2 → Output Layer
   (3 nodes)    (4 nodes)        (4 nodes)       (1 node)
```

## How it Works
1. **Forward Pass**: Data flows in one direction through layers
2. **Weighted Connections**: Each neuron connects to all neurons in next layer
3. **Activation Function**: Non-linear transformation at each neuron
4. **Backpropagation**: Errors propagate backward to update weights

## Mathematical Process
```
z = w·x + b           (Linear combination)
a = activation(z)     (Non-linear activation)
error = (a - y)²      (Loss calculation)
dw = -α · ∂error/∂w  (Gradient descent)
```

## Advantages ✅
- ✓ Simple and easy to understand
- ✓ Fast training and inference
- ✓ Works well for tabular/structured data
- ✓ Flexible architecture (can add/remove layers)
- ✓ Good baseline model

## Disadvantages ❌
- ✗ Cannot capture spatial relationships (images)
- ✗ Cannot handle sequential/temporal data efficiently
- ✗ Vanishing gradient problem in deep networks
- ✗ No memory (treats each input independently)
- ✗ Poor for non-Euclidean data (graphs, trees)

## Use Cases 🎯
- Tabular data classification/regression
- Time series prediction (with preprocessing)
- Pattern recognition
- Function approximation
- Binary/multi-class classification
- Approximators for any continuous function

In [ ]:
# FNN Implementation Example
from tensorflow.keras.datasets import mnist

# Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Flatten images to 1D
X_train = X_train.reshape(-1, 28*28).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28*28).astype('float32') / 255.0

# Build FNN model
fnn_model = Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

fnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("FNN Model Architecture:")
fnn_model.summary()

In [ ]:
# Train FNN
history = fnn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=0
)

# Evaluate
fnn_loss, fnn_acc = fnn_model.evaluate(X_test, y_test, verbose=0)
print(f"\nFNN Test Accuracy: {fnn_acc:.4f}")
print(f"FNN Test Loss: {fnn_loss:.4f}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('FNN: Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('FNN: Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

---
# 2. Convolutional Neural Networks (CNN) {#cnn}

## Architecture Overview
```
Input Image → [Conv → ReLU → Pool] × N → Flatten → Dense Layers → Output
  (32×32×3)      (5×5 filter)          (Fully Connected)
```

## How it Works
1. **Convolution**: Applies filters to detect features (edges, textures, shapes)
2. **Activation**: ReLU introduces non-linearity
3. **Pooling**: Downsampling reduces spatial dimensions
4. **Flattening**: Converts 2D to 1D for dense layers
5. **Dense Layers**: Classification/regression

## Mathematical Process
```
Conv Output = ReLU(Filter * Input + bias)
Max Pool = max(sliding window)
Feature Map Progression:
  Layer 1: Edges (horizontal, vertical, diagonal)
  Layer 2: Corners, textures
  Layer 3: Parts (eyes, wheels)
  Layer 4: Objects (faces, cars)
```

## Advantages ✅
- ✓ Excellent for image recognition
- ✓ Parameter sharing reduces model size
- ✓ Captures spatial relationships
- ✓ Hierarchical feature learning
- ✓ Translation invariance
- ✓ Proven state-of-the-art performance

## Disadvantages ❌
- ✗ High computational cost
- ✗ Requires large datasets
- ✗ Difficulty interpreting learned features
- ✗ Poor for sequential data
- ✗ Cannot capture temporal relationships
- ✗ Hyperparameter tuning complexity

## Use Cases 🎯
- Image classification (CIFAR-10, ImageNet)
- Object detection (YOLO, R-CNN)
- Face recognition & verification
- Medical image analysis
- Semantic segmentation
- Facial expression recognition

In [ ]:
# CNN Implementation Example
from tensorflow.keras.datasets import cifar10

# Load CIFAR-10 dataset
(X_train_cnn, y_train_cnn), (X_test_cnn, y_test_cnn) = cifar10.load_data()
X_train_cnn = X_train_cnn.astype('float32') / 255.0
X_test_cnn = X_test_cnn.astype('float32') / 255.0
y_train_cnn = keras.utils.to_categorical(y_train_cnn, 10)
y_test_cnn = keras.utils.to_categorical(y_test_cnn, 10)

# Build CNN model
cnn_model = Sequential([
    # First Conv Block
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Second Conv Block
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Dense layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("CNN Model Architecture:")
cnn_model.summary()

In [ ]:
# Train CNN
history_cnn = cnn_model.fit(
    X_train_cnn[:10000], y_train_cnn[:10000],  # Using subset for demo
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=0
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn[:2000], y_test_cnn[:2000], verbose=0)
print(f"CNN Test Accuracy: {cnn_acc:.4f}")
print(f"CNN Test Loss: {cnn_loss:.4f}")

---
# 3. Recurrent Neural Networks (RNN) {#rnn}

## Architecture Overview
```
Input Sequence: [x₁, x₂, x₃, x₄]
        ↓
    RNN Cell (recurrent connection)
      h₀ → h₁ → h₂ → h₃
      ↓    ↓    ↓    ↓
    Output: [y₁, y₂, y₃, y₄]
```

## How it Works
1. **Recurrent Connection**: Hidden state passes from t to t+1
2. **Sequential Processing**: Processes one element at a time
3. **Memory**: Current output depends on previous inputs via hidden state
4. **Weight Sharing**: Same weights across time steps
5. **BPTT**: Backpropagation Through Time for training

## Mathematical Process
```
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)
y_t = W_hy · h_t + b_y

Where:
  h_t = hidden state at time t
  W_hh = hidden-to-hidden weights
  W_xh = input-to-hidden weights
  W_hy = hidden-to-output weights
```

## Advantages ✅
- ✓ Handles variable-length sequences
- ✓ Captures temporal dependencies
- ✓ Memory of past inputs
- ✓ Parameter sharing across time
- ✓ Flexible output lengths

## Disadvantages ❌
- ✗ Vanishing gradient problem
- ✗ Difficulty learning long-term dependencies
- ✗ Computational cost (sequential processing)
- ✗ Hard to parallelize
- ✗ Training instability
- ✗ Exploding gradients

## Use Cases 🎯
- Time series prediction
- Language modeling
- Machine translation
- Stock price prediction
- Speech recognition
- Sentiment analysis

In [ ]:
# RNN Implementation Example - Time Series
from tensorflow.keras.datasets import imdb

# Load IMDB dataset
vocab_size = 10000
max_length = 100

(X_train_rnn, y_train_rnn), (X_test_rnn, y_test_rnn) = imdb.load_data(num_words=vocab_size)

# Pad sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train_rnn = pad_sequences(X_train_rnn, maxlen=max_length)
X_test_rnn = pad_sequences(X_test_rnn, maxlen=max_length)

# Build RNN model
rnn_model = Sequential([
    layers.Embedding(vocab_size, 32, input_length=max_length),
    layers.SimpleRNN(64, activation='relu', return_sequences=True),
    layers.SimpleRNN(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("RNN Model Architecture:")
rnn_model.summary()

In [ ]:
# Train RNN
history_rnn = rnn_model.fit(
    X_train_rnn[:5000], y_train_rnn[:5000],
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

rnn_loss, rnn_acc = rnn_model.evaluate(X_test_rnn[:2000], y_test_rnn[:2000], verbose=0)
print(f"RNN Test Accuracy: {rnn_acc:.4f}")
print(f"RNN Test Loss: {rnn_loss:.4f}")

---
# 4. Long Short-Term Memory (LSTM) {#lstm}

## Architecture Overview
```
                   ┌─────────────┐
                   │ Cell State  │
                   │ (Memory)    │
                   └──────┬──────┘
                          │
     ┌─────────┐  ┌──────────────┐  ┌─────────┐
     │ Forget  │  │   Input      │  │ Output  │
     │ Gate    │  │   Gate       │  │ Gate    │
     └─────────┘  └──────────────┘  └─────────┘
      (0-1)         (0-1)             (0-1)
```

## How it Works
1. **Forget Gate**: Decides what to forget from cell state
2. **Input Gate**: Decides what new info to add
3. **Update Cell State**: Add new information
4. **Output Gate**: Decides what to output

## Mathematical Process
```
f_t = sigmoid(W_f · [h_{t-1}, x_t] + b_f)     # Forget gate
i_t = sigmoid(W_i · [h_{t-1}, x_t] + b_i)     # Input gate
C̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)       # Candidate values
C_t = f_t * C_{t-1} + i_t * C̃_t              # Cell state
o_t = sigmoid(W_o · [h_{t-1}, x_t] + b_o)     # Output gate
h_t = o_t * tanh(C_t)                          # Hidden state
```

## Advantages ✅
- ✓ Solves vanishing gradient problem
- ✓ Captures long-term dependencies
- ✓ Better training stability
- ✓ Memory cell stores information
- ✓ Excellent for sequential data
- ✓ Industry standard for RNNs

## Disadvantages ❌
- ✗ More complex than vanilla RNN
- ✗ Higher computational cost
- ✗ More parameters to train
- ✗ Requires more data
- ✗ Slower inference than RNN
- ✗ Still sequential (hard to parallelize)

## Use Cases 🎯
- Language translation (seq2seq)
- Speech recognition
- Time series forecasting
- Stock price prediction
- Music generation
- Handwriting recognition
- Video action recognition

In [ ]:
# LSTM Implementation Example
lstm_model = Sequential([
    layers.Embedding(vocab_size, 32, input_length=max_length),
    layers.LSTM(64, activation='relu', return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("LSTM Model Architecture:")
lstm_model.summary()

In [ ]:
# Train LSTM
history_lstm = lstm_model.fit(
    X_train_rnn[:5000], y_train_rnn[:5000],
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test_rnn[:2000], y_test_rnn[:2000], verbose=0)
print(f"LSTM Test Accuracy: {lstm_acc:.4f}")
print(f"LSTM Test Loss: {lstm_loss:.4f}")

---
# 5. Gated Recurrent Unit (GRU) {#gru}

## Architecture Overview
```
Simpler than LSTM (2 gates instead of 3):

     ┌─────────┐  ┌──────────┐
     │ Reset   │  │ Update   │
     │ Gate    │  │ Gate     │
     └─────────┘  └──────────┘
      (0-1)       (0-1)
```

## How it Works
1. **Reset Gate**: Controls how much past info to discard
2. **Update Gate**: Controls how much to keep old state
3. **Candidate State**: Combines reset gate with new input
4. **New Hidden State**: Weighted combination of old and new

## Mathematical Process
```
r_t = sigmoid(W_r · [h_{t-1}, x_t] + b_r)      # Reset gate
z_t = sigmoid(W_z · [h_{t-1}, x_t] + b_z)      # Update gate
h̃_t = tanh(W · [r_t * h_{t-1}, x_t] + b)      # Candidate
h_t = (1 - z_t) * h_{t-1} + z_t * h̃_t         # New hidden state
```

## Advantages ✅
- ✓ Simpler than LSTM (fewer parameters)
- ✓ Faster training than LSTM
- ✓ Less memory required
- ✓ Handles long-term dependencies
- ✓ Easier to understand
- ✓ Good generalization

## Disadvantages ❌
- ✗ Less powerful than LSTM for complex tasks
- ✗ May underfit on large datasets
- ✗ Still sequential processing
- ✗ Not as well-studied as LSTM
- ✗ Limited parallelization

## Use Cases 🎯
- Machine translation
- Speech recognition
- Time series with limited data
- Sentiment analysis
- Quick prototyping
- Resource-constrained environments

In [ ]:
# GRU Implementation Example
gru_model = Sequential([
    layers.Embedding(vocab_size, 32, input_length=max_length),
    layers.GRU(64, activation='relu', return_sequences=True),
    layers.Dropout(0.2),
    layers.GRU(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("GRU Model Architecture:")
gru_model.summary()

In [ ]:
# Train GRU
history_gru = gru_model.fit(
    X_train_rnn[:5000], y_train_rnn[:5000],
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

gru_loss, gru_acc = gru_model.evaluate(X_test_rnn[:2000], y_test_rnn[:2000], verbose=0)
print(f"GRU Test Accuracy: {gru_acc:.4f}")
print(f"GRU Test Loss: {gru_loss:.4f}")

---
# 6. Autoencoders {#autoencoder}

## Architecture Overview
```
Input (784) → Dense(256) → Dense(128) → Dense(32) [Bottleneck]
                                             ↓
              Dense(128) ← Dense(64) ← Dense(32)
                 ↓
            Output (784)
```

## How it Works
1. **Encoder**: Compresses input to latent representation
2. **Bottleneck**: Low-dimensional representation (key features)
3. **Decoder**: Reconstructs output from bottleneck
4. **Loss**: Reconstruction error (Input vs Output)
5. **Learning**: Learns efficient data representation

## Mathematical Process
```
h = encoder(x)              # Encode to latent space
x̂ = decoder(h)              # Decode from latent space
Loss = ||x - x̂||²           # Reconstruction loss
Goal: Minimize reconstruction error
```

## Advantages ✅
- ✓ Unsupervised dimensionality reduction
- ✓ Feature extraction
- ✓ Anomaly detection
- ✓ Denoising capability
- ✓ Learns hierarchical representations
- ✓ No labels required

## Disadvantages ❌
- ✗ Reconstruction loss can be high
- ✗ Poor for high-dimensional data without deep architecture
- ✗ Training instability
- ✗ Prone to overfitting
- ✗ Difficult interpretation of latent space
- ✗ May learn trivial identity function

## Use Cases 🎯
- Image denoising
- Dimensionality reduction
- Anomaly detection
- Data compression
- Feature extraction
- Image restoration

In [ ]:
# Autoencoder Implementation
# Use MNIST data
(X_train_ae, _), (X_test_ae, _) = mnist.load_data()
X_train_ae = X_train_ae.reshape(-1, 28*28).astype('float32') / 255.0
X_test_ae = X_test_ae.reshape(-1, 28*28).astype('float32') / 255.0

# Add noise for denoising autoencoder
noise_factor = 0.5
X_train_noisy = X_train_ae + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train_ae.shape)
X_test_noisy = X_test_ae + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_test_ae.shape)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

# Build Autoencoder
autoencoder = Sequential([
    # Encoder
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),  # Bottleneck
    
    # Decoder
    layers.Dense(64, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(784, activation='sigmoid')  # Reconstruct 28x28
])

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

print("Autoencoder Architecture:")
autoencoder.summary()

In [ ]:
# Train Autoencoder
history_ae = autoencoder.fit(
    X_train_noisy[:10000], X_train_ae[:10000],  # Input: noisy, Target: clean
    epochs=5,
    batch_size=256,
    validation_split=0.2,
    verbose=0
)

# Test reconstruction
X_test_reconstructed = autoencoder.predict(X_test_noisy[:100], verbose=0)

# Visualize results
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for i in range(10):
    axes[0, i].imshow(X_test_ae[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(X_test_noisy[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(X_test_reconstructed[i].reshape(28, 28), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Noisy', fontsize=12)
axes[2, 0].set_ylabel('Reconstructed', fontsize=12)
plt.tight_layout()
plt.show()

print("Autoencoder reconstruction complete!")

---
# 7. Variational Autoencoders (VAE) {#vae}

## Architecture Overview
```
Input → Encoder → [Mean, LogVar] → Sample(μ, σ) → Decoder → Output
                   (Reparameterization Trick)
```

## How it Works
1. **Encoder**: Maps input to distribution parameters (mean, variance)
2. **Reparameterization**: z = μ + σ * ε (where ε ~ N(0,1))
3. **Latent Space**: Continuous, smooth distribution
4. **Decoder**: Reconstructs from sampled latent vector
5. **Loss**: Reconstruction + KL divergence

## Mathematical Process
```
Loss = Reconstruction Loss + KL Divergence

Reconstruction Loss = ||x - decoder(z)||²
KL Divergence = 0.5 * Σ(μ² + σ² - log(σ²) - 1)

z = μ + σ * ε  (ε ~ N(0,1))
```

## Advantages ✅
- ✓ Continuous, smooth latent space
- ✓ Can generate new samples
- ✓ Better theoretical foundation
- ✓ Enables interpolation in latent space
- ✓ Regularized latent representation
- ✓ Useful for semi-supervised learning

## Disadvantages ❌
- ✗ More complex than standard autoencoder
- ✗ Slower training
- ✗ KL divergence can vanish (posterior collapse)
- ✗ Blurry reconstruction
- ✗ Requires careful hyperparameter tuning
- ✗ More difficult to debug

## Use Cases 🎯
- Generative image synthesis
- Semi-supervised learning
- Data augmentation
- Image interpolation
- Representation learning
- Anomaly detection

In [ ]:
# Variational Autoencoder Implementation
from tensorflow.keras import Model
from tensorflow.keras.layers import Lambda
import tensorflow.keras.backend as K

latent_dim = 2  # 2D latent space for visualization
input_shape = 784

# Encoder
encoder_inputs = layers.Input(shape=(input_shape,))
x = layers.Dense(256, activation='relu')(encoder_inputs)
x = layers.Dense(128, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)

# Sampling layer (Reparameterization trick)
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])

# Decoder
decoder_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(decoder_inputs)
x = layers.Dense(256, activation='relu')(x)
decoder_outputs = layers.Dense(input_shape, activation='sigmoid')(x)
decoder = Model(decoder_inputs, decoder_outputs, name='decoder')

outputs = decoder(z)

# VAE model
vae = Model(encoder_inputs, outputs, name='vae')

# VAE Loss
reconstruction_loss = K.mean(K.binary_crossentropy(encoder_inputs, outputs))
kl_loss = -0.5 * K.mean(K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=1))
vae_loss = reconstruction_loss + kl_loss

vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

print("VAE Architecture:")
vae.summary()

In [ ]:
# Train VAE
history_vae = vae.fit(
    X_train_ae[:10000],
    epochs=5,
    batch_size=256,
    validation_split=0.2,
    verbose=0
)

print("VAE training complete!")

---
# 8. Generative Adversarial Networks (GAN) {#gan}

## Architecture Overview
```
Random Noise → Generator → Fake Images
                              ↓
Real Images ──────────────→ Discriminator → Real/Fake (0/1)
             Fake Images ──↗
```

## How it Works (Game Theory)
1. **Generator**: Creates fake images from random noise
2. **Discriminator**: Classifies real vs fake images
3. **Adversarial Training**:
   - Generator: Tries to fool discriminator
   - Discriminator: Tries to catch fake images
4. **Equilibrium**: Both improve together

## Mathematical Process
```
min_G max_D E_x[log D(x)] + E_z[log(1 - D(G(z)))]

Where:
  G(z) = Generator output (fake image)
  D(x) = Discriminator output (probability real)
  z = Random noise (latent vector)
  x = Real image
```

## Advantages ✅
- ✓ Generates realistic images
- ✓ No explicit loss function
- ✓ Powerful generative model
- ✓ Can create high-quality synthetic data
- ✓ Applications in style transfer
- ✓ Learns implicit distribution

## Disadvantages ❌
- ✗ Mode collapse (generates limited variety)
- ✗ Training instability
- ✗ Very difficult to train
- ✗ Vanishing gradient problem
- ✗ Requires careful hyperparameter tuning
- ✗ Evaluating quality is subjective
- ✗ Computationally expensive

## Use Cases 🎯
- Image generation (StyleGAN)
- Super-resolution
- Style transfer
- Image-to-image translation (Pix2Pix, CycleGAN)
- Data augmentation
- Face generation/synthesis

In [ ]:
# Simple GAN Implementation
latent_size = 100
img_shape = 784

# Generator
generator = Sequential([
    layers.Dense(256, activation='relu', input_dim=latent_size),
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(img_shape, activation='sigmoid')
])

# Discriminator
discriminator = Sequential([
    layers.Dense(512, activation='relu', input_dim=img_shape),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

discriminator.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# GAN
discriminator.trainable = False
gan = Sequential([generator, discriminator])
gan.compile(
    loss='binary_crossentropy',
    optimizer='adam'
)

print("Generator:")
generator.summary()
print("\nDiscriminator:")
discriminator.summary()

In [ ]:
# Train GAN (simplified)
batch_size = 64
epochs = 5

for epoch in range(epochs):
    # Train discriminator
    idx = np.random.randint(0, X_train_ae.shape[0], batch_size)
    real_images = X_train_ae[idx]
    
    noise = np.random.normal(0, 1, (batch_size, latent_size))
    fake_images = generator.predict(noise, verbose=0)
    
    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))
    d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size, 1)))
    
    # Train generator
    noise = np.random.normal(0, 1, (batch_size, latent_size))
    discriminator.trainable = False
    g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}, D Loss Real: {d_loss_real[0]:.4f}, D Loss Fake: {d_loss_fake[0]:.4f}, G Loss: {g_loss:.4f}')

print("\nGAN training complete!")

---
# 9. Transformer Networks {#transformer}

## Architecture Overview
```
Input Sequence
     ↓
Embedding + Positional Encoding
     ↓
┌────────────────────┐
│ Transformer Block  │
├────────────────────┤
│ Multi-Head Attn.   │  × N
│ Feed-Forward       │
│ LayerNorm + Skip   │
└────────────────────┘
     ↓
Output Sequence
```

## How it Works
1. **Attention**: Learns relationships between all positions simultaneously
2. **Multi-Head**: Multiple attention mechanisms in parallel
3. **Positional Encoding**: Adds position information
4. **Feed-Forward**: Non-linear transformation
5. **Residual Connections**: Skip connections prevent vanishing gradients

## Mathematical Process
```
Query = Input · W_Q
Key = Input · W_K
Value = Input · W_V

Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V

MultiHead = Concat(head_1, ..., head_h) · W_O
```

## Advantages ✅
- ✓ Parallel processing (much faster than RNN)
- ✓ Captures long-range dependencies
- ✓ No vanishing gradient problem
- ✓ Can be pre-trained on large datasets
- ✓ Works with variable-length sequences
- ✓ State-of-the-art on many tasks
- ✓ Easy to transfer learning

## Disadvantages ❌
- ✗ Quadratic memory complexity (O(n²))
- ✗ Requires large amounts of data
- ✗ Computationally expensive
- ✗ Difficult to train from scratch
- ✗ Positional encoding design choices
- ✗ High inference latency for long sequences

## Use Cases 🎯
- Machine translation (BERT, GPT)
- Language modeling
- Question answering
- Text summarization
- Named entity recognition
- Image classification (Vision Transformer)
- Time series forecasting

In [ ]:
# Transformer Block Implementation
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

# Build Transformer model for sequence classification
embed_dim = 32
num_heads = 2
ff_dim = 128

transformer_model = Sequential([
    layers.Embedding(vocab_size, embed_dim, input_length=max_length),
    TransformerBlock(embed_dim, num_heads, ff_dim),
    TransformerBlock(embed_dim, num_heads, ff_dim),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.1),
    layers.Dense(20, activation="relu"),
    layers.Dropout(0.1),
    layers.Dense(1, activation="sigmoid"),
])

transformer_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

print("Transformer Model Architecture:")
transformer_model.summary()

In [ ]:
# Train Transformer
history_transformer = transformer_model.fit(
    X_train_rnn[:5000], y_train_rnn[:5000],
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

transformer_loss, transformer_acc = transformer_model.evaluate(X_test_rnn[:2000], y_test_rnn[:2000], verbose=0)
print(f"Transformer Test Accuracy: {transformer_acc:.4f}")
print(f"Transformer Test Loss: {transformer_loss:.4f}")

---
# 10. Bidirectional RNN & Attention Mechanisms {#attention}

## Bidirectional RNN
```
Forward RNN:  x₁ → x₂ → x₃ → x₄
               ↓    ↓    ↓    ↓
            h₁_f h₂_f h₃_f h₄_f

Backward RNN: x₁ ← x₂ ← x₃ ← x₄
               ↓    ↓    ↓    ↓
            h₁_b h₂_b h₃_b h₄_b

Output: [h₁_f||h₁_b, h₂_f||h₂_b, h₃_f||h₃_b, h₄_f||h₄_b]
        (concatenated)
```

## How it Works
1. **Forward Pass**: Process sequence left-to-right
2. **Backward Pass**: Process sequence right-to-left
3. **Concatenation**: Combine both representations
4. **Context**: Each position has future AND past context

## Attention Mechanism
```
Query: What are we looking for?
Key: Where are the important features?
Value: What information do we extract?

Attention(Q, K, V) = softmax(Q·K^T / √d_k)·V
```

## Advantages ✅
- ✓ Uses both past and future context
- ✓ Better understanding of context
- ✓ Excellent for NLP tasks
- ✓ Attention provides interpretability
- ✓ Better long-range dependencies

## Disadvantages ❌
- ✗ Double computation (2 RNNs)
- ✗ Higher memory requirement
- ✗ Slightly slower training
- ✗ Attention weights can be hard to interpret

## Use Cases 🎯
- Machine translation
- Named entity recognition
- POS tagging
- Semantic role labeling
- Question answering

In [ ]:
# Bidirectional LSTM with Attention
bidirectional_model = Sequential([
    layers.Embedding(vocab_size, 64, input_length=max_length),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Attention(),  # Attention layer
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

bidirectional_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Bidirectional LSTM + Attention Architecture:")
bidirectional_model.summary()

In [ ]:
# Train Bidirectional Model
history_bi = bidirectional_model.fit(
    X_train_rnn[:5000], y_train_rnn[:5000],
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

bi_loss, bi_acc = bidirectional_model.evaluate(X_test_rnn[:2000], y_test_rnn[:2000], verbose=0)
print(f"Bidirectional LSTM Test Accuracy: {bi_acc:.4f}")
print(f"Bidirectional LSTM Test Loss: {bi_loss:.4f}")

---
# 11. Graph Neural Networks (GNN) {#gnn}

## Architecture Overview
```
Graph: Nodes connected by edges with features
       ◯---◯---◯
       |   |   |
       ◯---◯---◯
       |   |   |
       ◯---◯---◯

GNN: Feature propagation through edges
```

## How it Works
1. **Message Passing**: Nodes gather info from neighbors
2. **Aggregation**: Combine neighbor features
3. **Update**: Compute new node features
4. **Layers**: Stack multiple message passing layers
5. **Readout**: Pool to graph-level predictions

## Mathematical Process
```
h_i^(l+1) = activation(W^(l) · h_i^(l) + Σ_{j∈N(i)} W_msg^(l) · h_j^(l))

Where:
  h_i = node i feature
  N(i) = neighbors of node i
  W = learnable weights
```

## Advantages ✅
- ✓ Handles non-Euclidean data
- ✓ Captures graph structure
- ✓ Parameter sharing across nodes
- ✓ Inductive learning capability
- ✓ Flexible node/edge features
- ✓ Scalable to large graphs

## Disadvantages ❌
- ✗ Computationally expensive for large graphs
- ✗ Over-smoothing with deep networks
- ✗ Limited theoretical understanding
- ✗ Requires graph structure
- ✗ Can lose node/edge information

## Use Cases 🎯
- Social network analysis
- Molecular property prediction
- Knowledge graphs
- Recommendation systems
- Traffic prediction
- Protein interaction networks

In [ ]:
print("Graph Neural Networks (GNN) - Conceptual Example")
print("=" * 50)

# Simple graph representation
import networkx as nx

# Create sample graph
G = nx.karate_club_graph()

# Visualize
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=300)
nx.draw_networkx_edges(G, pos, alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title('Karate Club Network (Example Graph for GNN)')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print("\nNote: GNNs would learn features for each node")
print("considering the graph structure and edge connections.")

---
# 12. Capsule Networks {#capsule}

## Architecture Overview
```
Input → Conv Layer → Primary Capsules → Routing → Output Capsules → Classification
                     (Low-level features)  (Dynamic)
```

## How it Works
1. **Capsules**: Groups of neurons representing entity properties
2. **Vector Outputs**: Not scalar (like neurons), but vectors
3. **Dynamic Routing**: Routing algorithm determines connections
4. **Viewpoint Equivariance**: Capsules adapt to viewpoint changes
5. **Part-Whole Hierarchy**: Learns hierarchical relationships

## Advantages ✅
- ✓ Better interpretability
- ✓ Viewpoint invariance
- ✓ Learns part-whole relationships
- ✓ Better generalization
- ✓ Fewer parameters than CNNs

## Disadvantages ❌
- ✗ Computationally expensive
- ✗ Complex to implement
- ✗ Routing algorithm overhead
- ✗ Limited proven effectiveness
- ✗ Difficult to tune
- ✗ Not widely adopted yet

## Use Cases 🎯
- Image classification
- Object detection
- 3D shape recognition
- Pose estimation
- Entity relationship extraction

---
# 13. Boltzmann Machines {#boltzmann}

## Architecture Overview
```
Visible Layer (Inputs)
      ◯ ◯ ◯ ◯
       \|/|/|
        X X X
       /|\|\|
      ◯ ◯ ◯
Hidden Layer (Latent)

(Fully connected, undirected)
```

## How it Works
1. **Energy-Based Model**: System minimizes energy
2. **Stochastic Updates**: Probabilistic activation
3. **Gibbs Sampling**: MCMC for sampling
4. **Contrastive Divergence**: Training algorithm
5. **Binary Units**: Probabilistic neurons

## Mathematical Process
```
Energy(v, h) = -Σ v_i b_i - Σ h_j c_j - Σ_{i,j} v_i h_j W_{ij}
P(v, h) ∝ exp(-Energy(v, h))
```

## Advantages ✅
- ✓ Generative model
- ✓ Well-founded probability theory
- ✓ Can model complex distributions
- ✓ Useful for dimensionality reduction

## Disadvantages ❌
- ✗ Very slow training (Gibbs sampling)
- ✗ Difficult to scale
- ✗ Approximation errors in sampling
- ✗ Not used much in practice
- ✗ Inferior to modern deep learning
- ✗ Limited practical applications

## Use Cases 🎯
- Collaborative filtering
- Feature learning
- Dimensionality reduction
- Generative models (historical)
- Image retrieval

---
# 14. Neural Network Comparison & Selection Guide {#comparison}

## Performance Comparison on Test Datasets

In [ ]:
# Comparison of different architectures
results = {
    'FNN': fnn_acc,
    'RNN': rnn_acc,
    'LSTM': lstm_acc,
    'GRU': gru_acc,
    'Transformer': transformer_acc,
    'Bidirectional': bi_acc
}

# Sort by accuracy
sorted_results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
models = list(sorted_results.keys())
accuracies = list(sorted_results.values())
colors = plt.cm.viridis(np.linspace(0, 1, len(models)))

bars = ax.bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_xlabel('Neural Network Architecture', fontsize=12, fontweight='bold')
ax.set_title('Neural Network Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nAccuracy Ranking:")
for i, (model, acc) in enumerate(sorted_results.items(), 1):
    print(f"{i}. {model}: {acc:.4f}")

## Comprehensive Comparison Table

In [ ]:
import pandas as pd

# Create comprehensive comparison
comparison_data = {
    'Architecture': [
        'FNN',
        'CNN',
        'RNN',
        'LSTM',
        'GRU',
        'Autoencoder',
        'VAE',
        'GAN',
        'Transformer',
        'Bi-LSTM',
        'GNN',
        'Capsule'
    ],
    'Training Speed': [
        '⚡⚡⚡',
        '⚡',
        '⚡⚡',
        '⚡',
        '⚡⚡',
        '⚡⚡',
        '⚡',
        '⚡',
        '⚡',
        '⚡',
        '🐢',
        '🐢🐢'
    ],
    'Inference Speed': [
        '⚡⚡⚡',
        '⚡⚡',
        '⚡⚡',
        '⚡',
        '⚡⚡',
        '⚡⚡',
        '⚡',
        '⚡',
        '⚡',
        '⚡',
        '🐢',
        '🐢'
    ],
    'Memory Usage': [
        'Low',
        'High',
        'Medium',
        'High',
        'Medium',
        'High',
        'Very High',
        'Very High',
        'Very High',
        'High',
        'Very High',
        'High'
    ],
    'Data Requirements': [
        'Low',
        'High',
        'Medium',
        'High',
        'Medium',
        'High',
        'Very High',
        'Very High',
        'Very High',
        'High',
        'High',
        'Very High'
    ],
    'Best For': [
        'Tabular data',
        'Images',
        'Sequences',
        'Time series',
        'Quick prototyping',
        'Compression',
        'Generation',
        'Image synthesis',
        'NLP, Sequences',
        'NLP',
        'Graphs',
        'Images (hierarchical)'
    ],
    'Interpretability': [
        'Medium',
        'Low',
        'Low',
        'Low',
        'Low',
        'Low',
        'Low',
        'Very Low',
        'Medium (Attention)',
        'Medium',
        'Medium',
        'High'
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*150)
print("NEURAL NETWORK ARCHITECTURES COMPARISON")
print("="*150)
print(df_comparison.to_string(index=False))
print("="*150)

## Decision Tree: Which Architecture to Choose?

```
START: What type of data do you have?
  ├─ Tabular/Structured Data?
  │  └─ YES → Use FNN (Fast & Simple)
  │
  ├─ Images?
  │  ├─ Classification/Detection? → Use CNN
  │  ├─ Generate new images? → Use GAN
  │  └─ Unsupervised? → Use Autoencoder or VAE
  │
  ├─ Text/Sequential Data?
  │  ├─ Sentiment/Classification? → Use LSTM or GRU
  │  ├─ Translation/Complex task? → Use Transformer
  │  ├─ Bidirectional understanding needed? → Use Bi-LSTM
  │  └─ Fast & simple? → Use GRU
  │
  ├─ Time Series?
  │  ├─ Long dependencies? → Use LSTM
  │  ├─ Quick solution? → Use GRU
  │  └─ Very long sequences? → Use Transformer
  │
  ├─ Graph Data?
  │  └─ YES → Use GNN
  │
  └─ Need to Generate Data?
     ├─ Realistic images? → Use GAN
     ├─ Smooth latent space? → Use VAE
     └─ Compressed representation? → Use Autoencoder
```

## Quick Reference: Pros & Cons Summary

In [ ]:
# Summary of pros and cons
summary = {
    'Architecture': [
        'FNN',
        'CNN',
        'RNN',
        'LSTM',
        'GRU',
        'Autoencoder',
        'VAE',
        'GAN',
        'Transformer',
        'GNN',
        'Capsule'
    ],
    'Key Advantages': [
        'Simple, Fast',
        'Spatial features, SOTA',
        'Sequential data',
        'Long-term memory',
        'Faster than LSTM',
        'Unsupervised learning',
        'Smooth generation',
        'High-quality synthesis',
        'Parallel, SOTA NLP',
        'Graph-structured data',
        'Hierarchical learning'
    ],
    'Key Disadvantages': [
        'No spatial awareness',
        'High computation cost',
        'Vanishing gradients',
        'Complex, slow',
        'Less powerful',
        'High reconstruction loss',
        'Blurry outputs',
        'Training instability',
        'Quadratic complexity',
        'Computationally expensive',
        'Not widely adopted'
    ]
}

df_summary = pd.DataFrame(summary)
print("\n" + "="*120)
print("NEURAL NETWORK SUMMARY: PROS & CONS")
print("="*120)
print(df_summary.to_string(index=False))
print("="*120)

## Use Case Recommendations

In [ ]:
use_cases = {
    'Task': [
        'Image Classification',
        'Sentiment Analysis',
        'Machine Translation',
        'Time Series Prediction',
        'Image Generation',
        'Object Detection',
        'Named Entity Recognition',
        'Anomaly Detection',
        'Recommendation System',
        'Question Answering',
        'Stock Price Prediction',
        'Face Recognition'
    ],
    'Recommended Architecture': [
        'CNN',
        'LSTM / Transformer',
        'Transformer / Seq2Seq',
        'LSTM / GRU',
        'GAN / VAE',
        'CNN / YOLO / R-CNN',
        'Bi-LSTM / Transformer',
        'Autoencoder',
        'GNN / Collaborative Filtering',
        'Transformer / Bi-LSTM',
        'LSTM / GRU',
        'CNN / Siamese'
    ],
    'Why': [
        'Captures spatial patterns in images',
        'Handles variable-length sequences',
        'Parallel processing, excellent for NLP',
        'Remembers long-term dependencies',
        'Creates realistic synthetic data',
        'Detects and localizes objects',
        'Bidirectional context understanding',
        'Learns normal patterns, detects outliers',
        'Learns user-item relationships',
        'Attention mechanism finds answer locations',
        'Predicts future values from past',
        'Learns facial embeddings for verification'
    ]
}

df_usecases = pd.DataFrame(use_cases)
print("\n" + "="*150)
print("USE CASE RECOMMENDATIONS")
print("="*150)
print(df_usecases.to_string(index=False))
print("="*150)

## Training Tips & Best Practices

### 1. Data Preparation
- **Normalize/Standardize**: Use StandardScaler, MinMaxScaler
- **Train-Val-Test Split**: 70-20-10 or 60-20-20
- **Augmentation**: Especially important for images
- **Imbalanced Data**: Use class weights or oversampling

### 2. Model Selection
- **Start Simple**: FNN → CNN → LSTM → Transformer
- **Baseline First**: Implement simple model first
- **Complexity Gradually**: Add layers/parameters incrementally
- **Transfer Learning**: Use pre-trained models when possible

### 3. Training Strategy
- **Learning Rate**: Start with 0.001, adjust based on loss
- **Batch Size**: Try 32, 64, 128 (power of 2)
- **Epochs**: Monitor validation loss, use early stopping
- **Optimizer**: Adam (default), SGD (fine-tuning), RMSprop

### 4. Regularization
- **Dropout**: 0.2-0.5 to prevent overfitting
- **L1/L2**: Penalize large weights
- **Batch Normalization**: Stabilize training
- **Early Stopping**: Stop when validation metric plateaus

### 5. Hyperparameter Tuning
- **Grid Search**: Try all combinations
- **Random Search**: Faster exploration
- **Bayesian Optimization**: Smart parameter search
- **Cross-Validation**: 5-fold or 10-fold

### 6. Debugging
- **Overfit Small Batch**: Model should fit small dataset
- **Check Gradients**: Ensure backprop is working
- **Plot Loss Curves**: Look for oscillations, plateaus
- **Compare Predictions**: Analyze failure cases
- **Loss Sanity Check**: Random predictions should give baseline loss

## Performance Metrics by Task Type

### Classification
- **Accuracy**: (TP + TN) / Total
- **Precision**: TP / (TP + FP)
- **Recall**: TP / (TP + FN)
- **F1-Score**: 2 × (Precision × Recall) / (Precision + Recall)
- **AUC-ROC**: Area under curve for different thresholds
- **Confusion Matrix**: Shows all correct/incorrect predictions

### Regression
- **MSE**: Mean Squared Error
- **RMSE**: Root Mean Squared Error
- **MAE**: Mean Absolute Error
- **R²**: Coefficient of determination
- **MAPE**: Mean Absolute Percentage Error

### Generative Models
- **FID**: Frechet Inception Distance (quality)
- **IS**: Inception Score
- **Perceptual Loss**: Human-like quality
- **KL Divergence**: For VAE

### NLP
- **BLEU**: Machine translation quality
- **ROUGE**: Summarization quality
- **Perplexity**: Language model quality
- **Accuracy**: Classification metrics

---
## Final Summary: Neural Network Landscape

### Evolution & Popularity
```
2012: AlexNet (CNN) → Computer Vision Revolution
2014: VGGNet, GAN → Better architectures
2015: ResNet, LSTM improvements → Deeper networks
2016: Attention mechanism → Seq2Seq
2017: Transformer → "Attention is All You Need"
2018: BERT, GPT → Pre-training revolution
2019-2021: Large-scale models → GPT-2, GPT-3
2022-2023: Vision Transformers, Diffusion Models → Multimodal
```

### When to Use Each Architecture
| Scenario | Best Choice | Why |
|----------|------------|-----|
| Small tabular data | FNN | Fast, simple |
| Large images | CNN | Spatial patterns |
| Text classification | LSTM/Transformer | Sequential data |
| Machine translation | Transformer | Parallel processing |
| Time series | LSTM/GRU | Temporal dependencies |
| Graph data | GNN | Graph structure |
| Image generation | GAN/VAE | Generative task |
| Dimensionality reduction | Autoencoder | Unsupervised |
| Need interpretability | Capsule/Attention | Explainability |
| Real-time processing | Lightweight (GRU, MobileNet) | Speed |

### Key Takeaways
1. **No One-Size-Fits-All**: Choose based on your specific task
2. **Start Simple**: Don't jump to complex architectures
3. **Transfer Learning**: Use pre-trained models when available
4. **Data Quality > Model Complexity**: Better data beats fancier models
5. **Experiment & Iterate**: Try multiple architectures
6. **Monitor & Debug**: Keep track of training metrics
7. **Regularize**: Prevent overfitting with dropout, normalization
8. **Stay Updated**: Deep learning evolves rapidly!

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                   NEURAL NETWORKS GUIDE - COMPLETED                       ║
║                                                                           ║
║  You've learned:                                                         ║
║  ✓ 13 types of neural networks                                          ║
║  ✓ How each architecture works                                          ║
║  ✓ Mathematical foundations                                             ║
║  ✓ Advantages & disadvantages                                           ║
║  ✓ Real-world use cases                                                 ║
║  ✓ Practical implementation examples                                    ║
║  ✓ Performance comparisons                                              ║
║  ✓ Selection guidelines                                                 ║
║  ✓ Training best practices                                              ║
║                                                                           ║
║  Next Steps:                                                             ║
║  1. Choose a dataset matching your use case                             ║
║  2. Start with simple architecture                                      ║
║  3. Implement and train                                                 ║
║  4. Evaluate and iterate                                                ║
║  5. Deploy with monitoring                                              ║
║                                                                           ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")

print("\nModel Accuracies:")
for model, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model:20s}: {acc:.4f}")